In [32]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import librosa
import warnings
import pickle
warnings.filterwarnings('ignore')

In [16]:
# ==================== CONFIGURATION ====================
DATA_PATH = "C:/Users/ishap/PycharmProjects/ASE project/dataset"   # update if needed
SAMPLE_RATE = 22050
N_MELS = 128
USE_DELTAS = True
USE_AUGMENTATION = True          # includes SpecAugment + Mixup
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4              # L2 regularization
EARLY_STOPPING_PATIENCE = 15
VALIDATION_SPLIT = 0.15
MIXUP_ALPHA = 0.2                 # Beta distribution parameter for mixup

# Emotion mapping (RAVDESS)
emotions = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"
}

In [17]:
# ==================== COLLECT FILES ====================
file_data = []  # (path, emotion, actor_id)
for root, _, files in os.walk(DATA_PATH):
    for file in files:
        if not file.lower().endswith('.wav'):
            continue
        parts = file.split('-')
        if len(parts) >= 7 and parts[2] in emotions:
            emotion = emotions[parts[2]]
            actor = parts[6].split('.')[0]
            file_data.append((os.path.join(root, file), emotion, actor))

print(f"Found {len(file_data)} valid RAVDESS files.")
if len(file_data) == 0:
    raise ValueError("No valid files found. Check dataset path.")

Found 1440 valid RAVDESS files.


In [18]:
# ==================== ACTOR-BASED SPLIT ====================
actors = list(set(d[2] for d in file_data))
train_actors, test_actors = train_test_split(actors, test_size=0.2, random_state=42)
val_actors, test_actors = train_test_split(test_actors, test_size=0.5, random_state=42)

train_files, train_labels = [], []
val_files, val_labels = [], []
test_files, test_labels = [], []

for path, emotion, actor in file_data:
    if actor in train_actors:
        train_files.append(path)
        train_labels.append(emotion)
    elif actor in val_actors:
        val_files.append(path)
        val_labels.append(emotion)
    else:
        test_files.append(path)
        test_labels.append(emotion)

print(f"Train: {len(train_files)} files, Val: {len(val_files)} files, Test: {len(test_files)} files")

Train: 1140 files, Val: 120 files, Test: 180 files


In [19]:
# ==================== FEATURE EXTRACTION (MEL-SPECTROGRAM) ====================
def extract_mel_spectrogram(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS, fmax=8000)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    if USE_DELTAS:
        delta = librosa.feature.delta(log_mel)
        delta2 = librosa.feature.delta(log_mel, order=2)
        feat = np.concatenate((log_mel, delta, delta2), axis=0)
    else:
        feat = log_mel
    feat = feat.T   # (time, features)
    return feat

def spec_augment(feat, time_mask_param=10, freq_mask_param=5):
    feat = feat.copy()
    freq_dim = feat.shape[1]
    time_dim = feat.shape[0]
    # Frequency masking
    for _ in range(1):
        f = np.random.randint(0, freq_mask_param)
        f0 = np.random.randint(0, freq_dim - f)
        feat[:, f0:f0+f] = 0
    # Time masking
    for _ in range(1):
        t = np.random.randint(0, time_mask_param)
        t0 = np.random.randint(0, time_dim - t)
        feat[t0:t0+t, :] = 0
    return feat


In [20]:
# ==================== DATASET WITH NORMALIZATION ====================
class EmotionDataset(Dataset):
    def __init__(self, file_list, label_list, augment=False, norm_stats=None):
        self.file_list = file_list
        self.label_list = label_list
        self.augment = augment
        self.norm_stats = norm_stats
        self.data = []
        self._build()

    def _build(self):
        for path, lbl in zip(self.file_list, self.label_list):
            feat = extract_mel_spectrogram(path)
            if feat is None:
                continue
            self.data.append((feat, lbl))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        feat, label = self.data[idx]
        feat = torch.tensor(feat, dtype=torch.float32)
        if self.norm_stats is not None:
            mean, std = self.norm_stats
            feat = (feat - mean) / std
        if self.augment:
            feat = spec_augment(feat.numpy())
            feat = torch.tensor(feat, dtype=torch.float32)
        return feat, label

def compute_normalization_stats(dataset):
    all_feats = []
    for feat, _ in dataset.data:
        all_feats.append(feat)
    all_feats = np.concatenate(all_feats, axis=0)
    mean = np.mean(all_feats, axis=0)
    std = np.std(all_feats, axis=0) + 1e-8
    return torch.tensor(mean, dtype=torch.float32), torch.tensor(std, dtype=torch.float32)

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels

In [21]:
# ==================== CNN-LSTM MODEL WITH ATTENTION ====================
class EmotionCNN_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.4):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool = nn.MaxPool1d(2)
        self.dropout_cnn = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=128, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout
        )
        self.dropout_lstm = nn.Dropout(dropout)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        x = x.permute(0, 2, 1)   # (batch, input_dim, time)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_cnn(x)

        x = x.permute(0, 2, 1)   # (batch, new_time, 128)
        new_lengths = lengths // 4

        packed = pack_padded_sequence(x, new_lengths.cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)

        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        context = self.dropout_lstm(context)

        out = self.fc(context)
        return out

In [22]:
# ==================== ENCODE LABELS ====================
le = LabelEncoder()
le.fit(train_labels + val_labels + test_labels)

train_labels_enc = le.transform(train_labels)
val_labels_enc = le.transform(val_labels)
test_labels_enc = le.transform(test_labels)

In [23]:
# ==================== CLASS WEIGHTS ====================
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels_enc), y=train_labels_enc)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device if torch.cuda.is_available() else 'cpu')


In [24]:
# ==================== CREATE DATASETS AND DATALOADERS ====================
raw_train_dataset = EmotionDataset(train_files, train_labels_enc, augment=False, norm_stats=None)
mean, std = compute_normalization_stats(raw_train_dataset)

train_dataset = EmotionDataset(train_files, train_labels_enc, augment=USE_AUGMENTATION, norm_stats=(mean, std))
val_dataset = EmotionDataset(val_files, val_labels_enc, augment=False, norm_stats=(mean, std))
test_dataset = EmotionDataset(test_files, test_labels_enc, augment=False, norm_stats=(mean, std))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

sample_feat, _ = train_dataset[0]
input_dim = sample_feat.shape[1]
num_classes = len(le.classes_)
print(f"Input feature dimension: {input_dim}, Number of classes: {num_classes}")

Input feature dimension: 384, Number of classes: 8


In [25]:
# ==================== MIXUP FUNCTION ====================
def mixup_data(x, y, alpha=1.0):
    """Returns mixed inputs, pairs of targets, and lambda"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [26]:
# ==================== TRAINING SETUP ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionCNN_LSTM(input_dim=input_dim, hidden_dim=128, num_layers=2, num_classes=num_classes, dropout=0.4).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

best_val_loss = float('inf')
patience_counter = 0

In [27]:
# ==================== TRAINING LOOP WITH MIXUP ====================
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    for inputs, lengths, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        lengths = lengths.to('cpu')
        
        # Apply mixup if enabled
        if USE_AUGMENTATION and MIXUP_ALPHA > 0:
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, MIXUP_ALPHA)
            inputs, targets_a, targets_b = inputs.to(device), targets_a.to(device), targets_b.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs, lengths)
        
        if USE_AUGMENTATION and MIXUP_ALPHA > 0:
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, lengths, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            lengths = lengths.to('cpu')
            outputs = model(inputs, lengths)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_loss /= len(val_dataset)
    val_acc = 100.0 * correct / total

    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

Epoch 1/100 | Train Loss: 2.0427 | Val Loss: 1.8977 | Val Acc: 25.00%
Epoch 2/100 | Train Loss: 1.8577 | Val Loss: 1.8325 | Val Acc: 23.33%
Epoch 3/100 | Train Loss: 1.7116 | Val Loss: 1.6693 | Val Acc: 35.83%
Epoch 4/100 | Train Loss: 1.6515 | Val Loss: 1.4675 | Val Acc: 43.33%
Epoch 5/100 | Train Loss: 1.5615 | Val Loss: 1.4079 | Val Acc: 46.67%
Epoch 6/100 | Train Loss: 1.5451 | Val Loss: 1.6335 | Val Acc: 31.67%
Epoch 7/100 | Train Loss: 1.4679 | Val Loss: 1.6417 | Val Acc: 32.50%
Epoch 8/100 | Train Loss: 1.4297 | Val Loss: 1.2843 | Val Acc: 49.17%
Epoch 9/100 | Train Loss: 1.2432 | Val Loss: 1.4893 | Val Acc: 50.00%
Epoch 10/100 | Train Loss: 1.3831 | Val Loss: 1.3434 | Val Acc: 45.83%
Epoch 11/100 | Train Loss: 1.1595 | Val Loss: 1.5314 | Val Acc: 44.17%
Epoch 12/100 | Train Loss: 1.1180 | Val Loss: 1.2003 | Val Acc: 59.17%
Epoch 13/100 | Train Loss: 1.1354 | Val Loss: 1.8387 | Val Acc: 38.33%
Epoch 14/100 | Train Loss: 1.0062 | Val Loss: 1.3971 | Val Acc: 50.83%
Epoch 15/100 | 

In [28]:
# ==================== TEST EVALUATION ====================
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for inputs, lengths, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        lengths = lengths.to('cpu')
        outputs = model(inputs, lengths)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
test_accuracy = 100.0 * test_correct / test_total
print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")


Final Test Accuracy: 61.11%


In [35]:
emotion_labels = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
with open('emotion_labels.pkl', 'wb') as f:
    pickle.dump(emotion_labels, f)